# TP2 - Clasificación de Género Musical (Spotify)
### Tecnología Digital VI - Inteligencia Artificial

**Integrantes:** _(completar)_

Este notebook resuelve el problema de clasificación multiclase de género
musical de la competencia de Kaggle del TP2: predecir el género de una
canción (**6 géneros posibles**) a partir de métricas acústicas de
Spotify (`danceability`, `energy`, `tempo`, etc.).

El dataset fue **anonimizado por la cátedra**: no trae `track_id`,
`artists`, `album_name` ni `track_name`, y el orden de las filas fue
mezclado al azar. `test.csv` trae una columna `Id` puramente sintética
(secuencial) que sirve solo para que Kaggle empareje las predicciones —
no está presente en `train.csv`.

**Notebook autocontenido**: a diferencia de otras versiones de este
trabajo, acá todo el código (config, carga de datos, preprocesamiento,
métricas, modelos, tuning, evaluación y generación de la submission) está
en las celdas de este mismo notebook — no depende de ningún paquete
externo (`src/spotify_clf/` ni similar). Solo necesita, en la misma
carpeta o en `data/raw/` relativo a donde se ejecute:

- `train.csv`
- `test.csv`
- `sample_submission.csv` (opcional, se usa para validar el formato antes de guardar la submission final)

Estructura del notebook:

1. Configuración y funciones auxiliares (todo el código del pipeline)
2. Carga de datos
3. Análisis Exploratorio de Datos (EDA)
4. Preprocesamiento
5. Baseline
6. Modelos y búsqueda de hiperparámetros (optimizando F1-Macro)
7. Model Selection (comparación por cross-validation)
8. Métrica MAP@3 y comparación F1-Macro vs MAP@3
9. Optimización pensando en MAP@3
10. Discusión de negocio: qué métrica usar según el caso
11. Entrenamiento final y generación del CSV para Kaggle
12. Conclusiones finales


## 1. Configuración y funciones auxiliares

### 1.1 Imports

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


### 1.2 Configuración: paths y nombres de columnas

In [ ]:
def _find_data_file(filename: str) -> Path:
    """Busca el CSV en algunos lugares razonables: al lado del notebook,
    o en data/raw/ (relativo al cwd o subiendo un nivel, por si se corre
    desde una carpeta notebooks/). Devuelve la primera ruta que exista; si
    ninguna existe, devuelve la ruta por default (data/raw/<filename>) para
    que el error de "no encontrado" sea claro más abajo.
    """
    candidates = [
        Path("data/raw") / filename,
        Path("../data/raw") / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]


TRAIN_PATH = _find_data_file("train.csv")
TEST_PATH = _find_data_file("test.csv")
SAMPLE_SUBMISSION_PATH = _find_data_file("sample_submission.csv")

MODELS_DIR = Path("outputs/models")
SUBMISSIONS_DIR = Path("outputs/submissions")

# Id numérico secuencial, solo en test.csv, sin valor predictivo.
ID_COL = "Id"

# Columna objetivo: presente en train.csv, ausente en test.csv, y es el
# mismo nombre que exige el formato de submission de Kaggle ("Id,Expected").
TARGET_COL = "Expected"
SUBMISSION_TARGET_COL = TARGET_COL

# El dataset viene anonimizado: no hay columnas de texto libre
# (track_id/artists/album_name/track_name). Se deja vacía para que
# get_features_and_target() sea robusta igual si alguna llegara a estar.
TEXT_COLS: list[str] = []

# Features categóricas de baja cardinalidad (se codifican con one-hot).
# `key` puede valer -1 cuando Spotify no detectó tonalidad; con
# OneHotEncoder eso se trata como un nivel más, sin manejo especial.
CATEGORICAL_COLS = ["key", "mode", "time_signature", "explicit"]

# Features numéricas continuas (se escalan).
NUMERIC_COLS = [
    "popularity",
    "duration_ms",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

RANDOM_STATE = 42


### 1.3 Carga de datos

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró {path}. Descargá el archivo de la competencia de "
            "Kaggle y colocalo junto a este notebook o en data/raw/."
        )
    return pd.read_csv(path)


def load_train(path: Path = TRAIN_PATH) -> pd.DataFrame:
    return load_csv(path)


def load_test(path: Path = TEST_PATH) -> pd.DataFrame:
    return load_csv(path)


def get_features_and_target(
    df: pd.DataFrame,
    target_col: str = TARGET_COL,
    drop_cols: list[str] | None = None,
):
    """Separa un DataFrame de train en X (features) e y (target)."""
    if drop_cols is None:
        drop_cols = [ID_COL, *TEXT_COLS]
    y = df[target_col]
    X = df.drop(columns=[c for c in [target_col, *drop_cols] if c in df.columns])
    return X, y


def train_val_split(
    df: pd.DataFrame,
    target_col: str = TARGET_COL,
    test_size: float = 0.2,
    random_state: int = RANDOM_STATE,
):
    """Split estratificado por género, para que la validación local tenga
    la misma proporción de clases que el train.
    """
    return train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df[target_col]
    )


### 1.4 Preprocesamiento

In [ ]:
def build_preprocessor(
    numeric_cols: list[str] = NUMERIC_COLS,
    categorical_cols: list[str] = CATEGORICAL_COLS,
) -> ColumnTransformer:
    numeric_pipeline = Pipeline(steps=[("scaler", StandardScaler())])
    categorical_pipeline = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ],
        remainder="drop",
    )


### 1.5 Métricas: F1-Macro y MAP@3

MAP@3 (Mean Average Precision at K=3): para cada canción, se le da
crédito al modelo si el género correcto aparece entre sus top-3
predicciones (ordenadas por probabilidad), con más peso cuanto más
arriba en el ranking esté. A diferencia de F1-Macro, no exige acertar en
la primera predicción.

In [ ]:
def average_precision_at_k(true_label, predicted_labels, k: int = 3) -> float:
    """Average precision @ k para una sola observación: 1/posición si
    `true_label` está entre las primeras k de `predicted_labels`, 0 si no.
    """
    predicted_labels = list(predicted_labels)[:k]
    for i, label in enumerate(predicted_labels):
        if label == true_label:
            return 1.0 / (i + 1)
    return 0.0


def mean_average_precision_at_k(y_true, y_pred_ranked, k: int = 3) -> float:
    """MAP@k sobre un conjunto de observaciones. `y_pred_ranked` trae, para
    cada observación, las clases predichas ordenadas de mayor a menor
    probabilidad.
    """
    scores = [
        average_precision_at_k(true, ranked, k=k)
        for true, ranked in zip(y_true, y_pred_ranked)
    ]
    return float(np.mean(scores))


def top_k_predictions(model, X, k: int = 3):
    """Top-k clases (ordenadas por probabilidad descendente) para cada fila
    de X, usando predict_proba + classes_ del modelo.
    """
    proba = model.predict_proba(X)
    classes = np.asarray(model.classes_)
    top_k_idx = np.argsort(proba, axis=1)[:, ::-1][:, :k]
    return classes[top_k_idx]


def map_at_3_from_model(model, X, y_true, k: int = 3) -> float:
    ranked = top_k_predictions(model, X, k=k)
    return mean_average_precision_at_k(y_true, ranked, k=k)


def f1_macro_scorer(estimator, X, y):
    """Scorer de F1-Macro compatible con GridSearchCV/RandomizedSearchCV/
    cross_validate. Se implementa como función plana (en vez de
    sklearn.metrics.make_scorer) porque en algunas instalaciones de
    scikit-learn ese scorer falla silenciosamente dentro de la búsqueda
    paralela (sklearn lo tapa devolviendo NaN en vez de propagar el
    error), lo que puede terminar "eligiendo" el peor modelo sin ningún
    aviso.
    """
    return f1_score(y, estimator.predict(X), average="macro")


def map_at_3_scorer(estimator, X, y):
    """Scorer de MAP@3 compatible con la misma API. Requiere que el
    estimator soporte predict_proba.
    """
    return map_at_3_from_model(estimator, X, y, k=3)


### 1.6 Registro de modelos y grillas de hiperparámetros

In [ ]:
@dataclass
class ModelSpec:
    name: str
    pipeline: Pipeline
    param_grid: dict[str, Any]


def _make_pipeline(classifier) -> Pipeline:
    return Pipeline(steps=[("preprocessor", build_preprocessor()), ("classifier", classifier)])


def get_model_registry(random_state: int = RANDOM_STATE) -> dict[str, ModelSpec]:
    registry: dict[str, ModelSpec] = {}

    # --- Baseline lineal ---------------------------------------------
    registry["logistic_regression"] = ModelSpec(
        name="logistic_regression",
        pipeline=_make_pipeline(LogisticRegression(max_iter=2000, random_state=random_state)),
        param_grid={
            "classifier__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
            "classifier__class_weight": [None, "balanced"],
        },
    )

    # --- Ensamble de árboles (bagging) --------------------------------
    registry["random_forest"] = ModelSpec(
        name="random_forest",
        pipeline=_make_pipeline(RandomForestClassifier(random_state=random_state, n_jobs=-1)),
        param_grid={
            "classifier__n_estimators": [200, 400, 600, 800],
            "classifier__max_depth": [None, 10, 20, 30, 40],
            "classifier__min_samples_leaf": [1, 2, 4, 8],
            "classifier__max_features": ["sqrt", "log2", None],
            "classifier__class_weight": [None, "balanced"],
        },
    )

    # --- Boosting (nativo de sklearn, sin dependencias extra) ---------
    registry["hist_gradient_boosting"] = ModelSpec(
        name="hist_gradient_boosting",
        pipeline=_make_pipeline(HistGradientBoostingClassifier(random_state=random_state)),
        param_grid={
            "classifier__learning_rate": [0.01, 0.03, 0.1, 0.3],
            "classifier__max_leaf_nodes": [15, 31, 63, 127],
            "classifier__max_iter": [100, 200, 300, 500],
            "classifier__l2_regularization": [0.0, 0.1, 1.0, 10.0],
            "classifier__min_samples_leaf": [10, 20, 30],
        },
    )

    # --- Basado en distancia (útil para contrastar con los anteriores) -
    registry["knn"] = ModelSpec(
        name="knn",
        pipeline=_make_pipeline(KNeighborsClassifier()),
        param_grid={
            "classifier__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 41, 51],
            "classifier__weights": ["uniform", "distance"],
            "classifier__p": [1, 2],  # 1 = distancia Manhattan, 2 = Euclídea
        },
    )

    return registry


### 1.7 Búsqueda de hiperparámetros (tuning)

Optimiza la métrica oficial de la competencia (F1-Macro) por default,
usando `StratifiedKFold` para preservar la proporción de géneros en cada
fold.

In [ ]:
def tune_model(
    spec: ModelSpec,
    X,
    y,
    search_type: Literal["grid", "random"] = "random",
    n_iter: int = 20,
    cv_folds: int = 5,
    scoring=f1_macro_scorer,
    random_state: int = RANDOM_STATE,
    n_jobs: int = -1,
    verbose: int = 1,
):
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)

    if search_type == "grid":
        search = GridSearchCV(
            estimator=spec.pipeline,
            param_grid=spec.param_grid,
            scoring=scoring,
            cv=cv,
            n_jobs=n_jobs,
            verbose=verbose,
            refit=True,
        )
    elif search_type == "random":
        search = RandomizedSearchCV(
            estimator=spec.pipeline,
            param_distributions=spec.param_grid,
            n_iter=n_iter,
            scoring=scoring,
            cv=cv,
            n_jobs=n_jobs,
            verbose=verbose,
            random_state=random_state,
            refit=True,
        )
    else:
        raise ValueError(f"search_type debe ser 'grid' o 'random', recibido: {search_type}")

    search.fit(X, y)

    if search.best_score_ != search.best_score_:  # NaN check sin importar numpy
        raise RuntimeError(
            f"El mejor score de la búsqueda para '{spec.name}' es NaN. Esto suele "
            "indicar que el scorer está fallando dentro de la búsqueda (sklearn lo "
            "tapa como NaN en vez de propagar el error). Probá `error_score='raise'` "
            "en GridSearchCV/RandomizedSearchCV para ver el traceback real."
        )

    return search


### 1.8 Model Selection: comparación por cross-validation

In [ ]:
@dataclass
class EvalResult:
    model_name: str
    f1_macro_mean: float
    f1_macro_std: float
    map_at_3_mean: float
    map_at_3_std: float
    fold_scores: dict = field(default_factory=dict)


def cross_validate_model(
    name: str,
    pipeline,
    X,
    y,
    cv_folds: int = 5,
    random_state: int = RANDOM_STATE,
    n_jobs: int = -1,
) -> EvalResult:
    """Evalúa un pipeline con F1-Macro y MAP@3 en la misma partición de
    cross-validation, para que ambas métricas sean directamente comparables.
    """
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    scoring = {"f1_macro": f1_macro_scorer, "map_at_3": map_at_3_scorer}
    scores = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring, n_jobs=n_jobs, return_train_score=False
    )

    if pd.isna(scores["test_f1_macro"]).any() or pd.isna(scores["test_map_at_3"]).any():
        raise RuntimeError(
            f"cross_validate_model('{name}') dio NaN en algún fold. Probá n_jobs=1 "
            "para ver el traceback real antes de confiar en esta comparación."
        )

    return EvalResult(
        model_name=name,
        f1_macro_mean=scores["test_f1_macro"].mean(),
        f1_macro_std=scores["test_f1_macro"].std(),
        map_at_3_mean=scores["test_map_at_3"].mean(),
        map_at_3_std=scores["test_map_at_3"].std(),
        fold_scores=scores,
    )


def results_to_dataframe(results: list[EvalResult]) -> pd.DataFrame:
    return (
        pd.DataFrame(
            [
                {
                    "model": r.model_name,
                    "f1_macro_mean": r.f1_macro_mean,
                    "f1_macro_std": r.f1_macro_std,
                    "map_at_3_mean": r.map_at_3_mean,
                    "map_at_3_std": r.map_at_3_std,
                }
                for r in results
            ]
        )
        .sort_values("f1_macro_mean", ascending=False)
        .reset_index(drop=True)
    )


### 1.9 Generación de la submission para Kaggle

In [ ]:
def build_submission(
    ids: pd.Series,
    predictions,
    id_col: str = ID_COL,
    target_col: str = SUBMISSION_TARGET_COL,
) -> pd.DataFrame:
    return pd.DataFrame({id_col: ids.values, target_col: predictions})


def save_submission(
    submission_df: pd.DataFrame,
    path: Path = SUBMISSIONS_DIR / "submission.csv",
) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    submission_df.to_csv(path, index=False)
    return path


def validate_submission_format(
    submission_df: pd.DataFrame,
    sample_path: Path = SAMPLE_SUBMISSION_PATH,
    id_col: str = ID_COL,
    target_col: str = SUBMISSION_TARGET_COL,
) -> None:
    """Chequea la submission contra sample_submission.csv (si existe):
    mismas columnas, misma cantidad de filas, mismo set de Id, sin nulos.
    """
    if not Path(sample_path).exists():
        return

    sample_df = pd.read_csv(sample_path)

    assert list(submission_df.columns) == list(sample_df.columns), (
        f"Columnas esperadas {list(sample_df.columns)}, encontradas {list(submission_df.columns)}"
    )
    assert len(submission_df) == len(sample_df), (
        f"Se esperaban {len(sample_df)} filas, hay {len(submission_df)}"
    )
    assert set(submission_df[id_col]) == set(sample_df[id_col]), (
        "El set de Id de la submission no coincide con el de sample_submission.csv"
    )
    assert submission_df[target_col].notna().all(), f"Hay valores nulos en la columna '{target_col}'"


## 2. Carga de datos

Coloquen `train.csv` y `test.csv` de **la competencia de Kaggle del TP**
junto a este notebook, o en `data/raw/` (ambas ubicaciones se detectan
automáticamente).

In [ ]:
train_df = load_train()
test_df = load_test()
train_df.shape, train_df.head()


In [ ]:
train_df.info()


In [ ]:
train_df.isna().sum()


## 3. Análisis Exploratorio de Datos (EDA)

Objetivo: entender qué características diferencian mejor a los distintos
géneros, y detectar problemas de calidad de datos (duplicados, outliers,
escalas muy distintas entre features) que condicionen el preprocesamiento.

### 3.1 Distribución de clases (géneros)

In [ ]:
genre_counts = train_df[TARGET_COL].value_counts()
print(f"Cantidad de géneros: {genre_counts.shape[0]}")
print(genre_counts)

fig, ax = plt.subplots(figsize=(8, 5))
genre_counts.plot(kind="bar", ax=ax)
ax.set_title("Cantidad de canciones por género")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()


**A tener en cuenta:** si algunos géneros tienen muchas menos canciones
que otros, F1-Macro los penaliza igual que a los géneros mayoritarios
(promedia el F1 por clase sin ponderar por soporte) — vale la pena mirar
el balance de clases antes de elegir `class_weight` o técnicas de
resampling.

### 3.2 Duplicados

In [ ]:
n_exact_dupes = train_df.duplicated().sum()
n_dupes_sin_target = train_df.drop(columns=[TARGET_COL]).duplicated().sum()
print(f"Filas exactamente duplicadas (incluyendo género): {n_exact_dupes}")
print(f"Filas con las mismas features pero posiblemente distinto género: {n_dupes_sin_target}")


Como la cátedra anonimizó el dataset (sin `track_id`, sin nombres), no
podemos identificar canciones repetidas por id. Lo que sí podemos
detectar son filas con **exactamente las mismas features** — si aparecen
con géneros distintos, es una señal de que dos canciones (o la misma,
indistinguible ya sin id) quedaron etiquetadas de forma diferente, lo
cual mete ruido irreducible en el problema (ningún modelo puede
distinguir dos filas idénticas con etiquetas distintas).

### 3.3 Distribución y correlación de features numéricas

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.flat, NUMERIC_COLS):
    sns.histplot(train_df[col], ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
corr = train_df[NUMERIC_COLS].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación entre features numéricas")
plt.tight_layout()
plt.show()


### 3.4 ¿Qué features separan mejor a los géneros?

Con solo 6 géneros podemos comparar directamente la distribución de las
features clave para todas las clases a la vez.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, ["danceability", "energy", "acousticness"]):
    sns.boxplot(data=train_df, x=TARGET_COL, y=feat, ax=ax)
    ax.set_title(f"{feat} por género")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=train_df, x="energy", y="acousticness", hue=TARGET_COL, alpha=0.4, s=15, ax=ax)
ax.set_title("energy vs acousticness por género")
plt.tight_layout()
plt.show()


**Conclusiones del EDA** _(completar con lo que observen en los gráficos
de arriba)_:

- ¿Qué features separan mejor a los géneros más "extremos" (ej. classical
  vs. metal/rock)?
- ¿Hay géneros que se solapan mucho entre sí en el espacio de features
  (candidatos naturales a confundirse, y por eso buenos candidatos a
  aparecer en el top-3 de MAP@3 aunque no sean la predicción top-1)?
- ¿Alguna feature tiene una distribución muy sesgada que amerite
  transformarla (log, etc.)?

## 4. Preprocesamiento

- **Features numéricas continuas** → `StandardScaler`. Necesario para
  modelos lineales (Logistic Regression) y basados en distancia (KNN); no
  perjudica a los modelos de árboles.
- **Features categóricas de baja cardinalidad** (`key`, `mode`,
  `time_signature`, `explicit`) → `OneHotEncoder`. Son códigos sin una
  relación de orden lineal real (ej. `key` es un pitch class 0-11), así
  que tratarlas como numéricas introduciría una relación de orden
  artificial.
- **`Id`** (solo en `test.csv`) → se descarta como feature.

El preprocesador se arma **dentro** de cada `Pipeline` (ver
`get_model_registry`), así que el `fit` del scaler/one-hot se hace solo
con datos de train en cada fold de cross-validation (evita data leakage).

In [ ]:
train_sub, val_sub = train_val_split(train_df, test_size=0.2)
X_train, y_train = get_features_and_target(train_sub)
X_val, y_val = get_features_and_target(val_sub)

preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
print("Shape de X_train luego del preprocesamiento:", X_train_transformed.shape)


## 5. Baseline

Antes de entrenar modelos "reales", establecemos un piso de comparación
con un clasificador trivial.

In [ ]:
baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_val)
baseline_f1 = f1_score(y_val, baseline_preds, average="macro")
print(f"Baseline (DummyClassifier) F1-Macro en validación: {baseline_f1:.4f}")


## 6. Modelos + búsqueda de hiperparámetros (F1-Macro)

Entrenamos 4 modelos de familias distintas (lineal, ensamble de árboles,
boosting, basado en distancia), cada uno con `RandomizedSearchCV`
optimizando **F1-Macro** (la métrica oficial de la competencia).

In [ ]:
registry = get_model_registry()
print("Modelos disponibles:", list(registry.keys()))

models_to_run = ["logistic_regression", "random_forest", "hist_gradient_boosting", "knn"]
search_results = {}

for model_name in models_to_run:
    spec = registry[model_name]
    print(f"\n=== Tuning {model_name} ===")
    search = tune_model(spec, X_train, y_train, search_type="random", n_iter=15, cv_folds=5)
    search_results[model_name] = search
    print(f"Mejor F1-Macro (CV): {search.best_score_:.4f}")
    print(f"Mejores hiperparámetros: {search.best_params_}")


In [ ]:
for model_name, search in search_results.items():
    preds = search.best_estimator_.predict(X_val)
    val_f1 = f1_score(y_val, preds, average="macro")
    print(f"{model_name}: F1-Macro en validación = {val_f1:.4f}")


## 7. Model Selection (cross-validation)

Comparamos los mejores estimadores de cada familia con cross-validation
sobre el set de train completo, para una estimación más robusta de
F1-Macro antes de elegir el modelo final.

In [ ]:
X_full, y_full = get_features_and_target(train_df)

eval_results = [
    cross_validate_model(name, search.best_estimator_, X_full, y_full, cv_folds=5)
    for name, search in search_results.items()
]

comparison_df = results_to_dataframe(eval_results)
comparison_df


El modelo con mayor `f1_macro_mean` (y desvío estándar razonable entre
folds) es el candidato a elegir para la competencia de Kaggle. Documenten
acá cuál eligieron y por qué (F1 medio, estabilidad entre folds, tiempo de
entrenamiento, interpretabilidad, etc.).

## 8. MAP@3: comparación con F1-Macro

`cross_validate_model` ya calculó MAP@3 en la tabla anterior
(`map_at_3_mean`) usando la misma partición de CV que F1-Macro, así que
son directamente comparables.

In [ ]:
comparison_df[["model", "f1_macro_mean", "map_at_3_mean"]]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison_df))
width = 0.35
ax.bar(x - width / 2, comparison_df["f1_macro_mean"], width, label="F1-Macro")
ax.bar(x + width / 2, comparison_df["map_at_3_mean"], width, label="MAP@3")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["model"], rotation=20)
ax.set_ylabel("score")
ax.set_title("F1-Macro vs MAP@3 por modelo")
ax.legend()
plt.tight_layout()
plt.show()


**Análisis esperado** _(completar)_: ¿el ranking de modelos cambia según
la métrica? Es razonable que MAP@3 sea sistemáticamente más alto que
F1-Macro (le alcanza con acertar en el top-3), pero lo interesante es si
**el modelo elegido por F1-Macro también es el mejor en MAP@3**, o si
conviene un modelo distinto según el objetivo de negocio.

## 9. Optimizando pensando en MAP@3

Algunas estrategias para mejorar MAP@3 aunque implique resignar algo de
F1-Macro:

1. **Calibrar las probabilidades** (`CalibratedClassifierCV`).
2. **`class_weight="balanced"`** (o resampling).
3. **Ensamble por promedio de probabilidades** entre modelos de familias
   distintas.
4. **Elegir el modelo/hiperparámetros por MAP@3 en vez de F1-Macro** en la
   búsqueda, usando `scoring=map_at_3_scorer` en `tune_model`.

In [ ]:
best_model_name = comparison_df.iloc[0]["model"]
spec = registry[best_model_name]

search_map3 = tune_model(
    spec, X_train, y_train, search_type="random", n_iter=15, cv_folds=5, scoring=map_at_3_scorer
)

print(f"Mejor MAP@3 (CV) optimizando directamente MAP@3: {search_map3.best_score_:.4f}")
print(f"Hiperparámetros: {search_map3.best_params_}")

model_f1_opt = search_results[best_model_name].best_estimator_
model_map3_opt = search_map3.best_estimator_

for label, model in [("optimizado para F1-Macro", model_f1_opt), ("optimizado para MAP@3", model_map3_opt)]:
    val_preds = model.predict(X_val)
    val_f1 = f1_score(y_val, val_preds, average="macro")
    val_map3 = map_at_3_from_model(model, X_val, y_val)
    print(f"{label}: F1-Macro={val_f1:.4f} | MAP@3={val_map3:.4f}")


**Conclusión** _(completar)_: ¿vale la pena, para este dataset, resignar
F1-Macro a cambio de MAP@3? ¿Cuánto se gana/pierde en cada métrica?

## 10. Discusión: ¿qué métrica usar según el caso de negocio?

_(Desarrollar con sus propias conclusiones — puntos de partida abajo)_

- **F1-Macro (top-1 forzado)** tiene sentido cuando el sistema debe tomar
  una única decisión automática sin intervención humana y todos los
  errores son igual de costosos (ej. etiquetar metadata interna de
  catálogo, moderación de contenido, clasificación para reporting).
- **MAP@3 (ranking, top-3)** es más fiel al caso de uso real de un
  sistema de recomendación: al usuario se le muestra una lista de
  opciones, y que el género correcto esté en el top-3 ya genera una buena
  experiencia, aunque no sea la primera opción mostrada.
- Otras métricas a considerar según el caso real de negocio:
  - **Precision@k / Recall@k** si interesa medir específicamente cuántas
    de las top-k recomendaciones son relevantes.
  - **NDCG** si, además de si el género correcto está en el top-k,
    importa en qué posición exacta.
  - **F1 ponderado (weighted)** si algunos géneros son más importantes
    para el negocio que otros.
  - **Métricas de negocio directas** (engagement, skip-rate) si se
    pudiera hacer un A/B test: F1-Macro y MAP@3 son proxies de
    laboratorio.

## 11. Entrenamiento final y submission para Kaggle

Reentrenamos el modelo elegido sobre **todo** `train.csv` y generamos el
CSV con el formato exacto que pide la consigna: `Id,Expected`.

Recuerden que tienen **máximo 5 envíos por día por equipo** — usen su
propia validación local (secciones 6-9) para decidir qué modelo mandar.

In [ ]:
final_model_name = best_model_name  # cambiar si se elige otro modelo
final_pipeline = registry[final_model_name].pipeline.set_params(
    **search_results[final_model_name].best_params_
)
final_pipeline.fit(X_full, y_full)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, MODELS_DIR / f"{final_model_name}_final.joblib")
print(f"Modelo final ({final_model_name}) entrenado sobre todo el train y guardado.")


In [ ]:
X_test = test_df.drop(columns=[c for c in [ID_COL, *TEXT_COLS] if c in test_df.columns])
test_predictions = final_pipeline.predict(X_test)

submission_df = build_submission(test_df[ID_COL], test_predictions)
validate_submission_format(submission_df)

out_path = save_submission(submission_df)
print(f"Submission guardada en: {out_path}")
submission_df.head()


## 12. Conclusiones finales

_(completar)_: resumen de decisiones de preprocesamiento, modelo elegido,
resultado en la competencia de Kaggle (F1-Macro público/privado), y la
recomendación final sobre F1-Macro vs. MAP@3 para el caso de negocio de
Spotify.